# Phase 6 Simplified Inspection Notebook

This notebook provides a concise, run-consistent QA pass for Phase 6 post-processing outputs.

Scope:
- Load the latest postprocess artifacts for one run label
- Validate goalie ledger row-type and formula integrity
- Validate faceoff and penalty inverse-conservation behavior
- Produce a compact pass/fail summary

In [7]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [8]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "Results").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


BASE_DIR = find_repo_root(Path.cwd())
RESULTS_ROOT = BASE_DIR / "Results" / "Transformer_xT"

# Set this explicitly when auditing a specific run.
RUN_LABEL_OVERRIDE: str | None = "Events_With_Saves"

if RUN_LABEL_OVERRIDE and RUN_LABEL_OVERRIDE.strip():
    RUN_LABEL = RUN_LABEL_OVERRIDE.strip()
else:
    run_dirs = [p for p in RESULTS_ROOT.iterdir() if p.is_dir()]
    run_dirs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if not run_dirs:
        raise FileNotFoundError(f"No run directories found under {RESULTS_ROOT}")
    RUN_LABEL = run_dirs[0].name

RUN_RESULTS_DIR = RESULTS_ROOT / RUN_LABEL
RUN_LOGS_DIR = RUN_RESULTS_DIR / "logs"

print(f"Base dir: {BASE_DIR}")
print(f"Run label: {RUN_LABEL}")
print(f"Run results dir: {RUN_RESULTS_DIR}")

Base dir: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO
Run label: Events_With_Saves
Run results dir: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Results\Transformer_xT\Events_With_Saves


In [9]:
summary_path = RUN_LOGS_DIR / "phase6_postprocess_summary.json"
adjusted_path = RUN_RESULTS_DIR / "oof_phase6_adjusted_predictions.parquet"
goalie_path = RUN_RESULTS_DIR / "optimus_reim_goalie_credit_ledger_events_only.parquet"
player_path = RUN_RESULTS_DIR / "optimus_reim_global_player_credit_ledger_events_only.parquet"

for p in [summary_path, adjusted_path, goalie_path, player_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact: {p}")

with summary_path.open("r", encoding="utf-8") as f:
    summary = json.load(f)

adjusted_df = pd.read_parquet(adjusted_path)
goalie_df = pd.read_parquet(goalie_path)
player_df = pd.read_parquet(player_path)

rows_summary = summary.get("rows", {}) if isinstance(summary.get("rows"), dict) else {}
display(pd.DataFrame([
    {
        "run_label": RUN_LABEL,
        "adjusted_rows": len(adjusted_df),
        "goalie_rows": len(goalie_df),
        "player_rows": len(player_df),
        "summary_adjusted_rows": rows_summary.get("adjusted"),
        "summary_goalie_rows": rows_summary.get("goalie_ledger"),
        "summary_goalie_row_type_counts": rows_summary.get("goalie_row_type_counts"),
    }
]))

,run_label,adjusted_rows,goalie_rows,player_rows,summary_adjusted_rows,summary_goalie_rows,summary_goalie_row_type_counts
0,Events_With_Saves,1657118,30470,1657118,1657118,30470,"{'save_component': 27824, 'goal_penalty': 2646}"


In [10]:
goalie_df = goalie_df.copy()

for c in ["xT_For", "xT_Against", "Net_xT", "goal_penalty_xT_Against", "native_save_xT_Against", "extra_freeze_credit_xT_Against"]:
    if c in goalie_df.columns:
        goalie_df[c] = pd.to_numeric(goalie_df[c], errors="coerce")

row_type_counts = goalie_df.get("goalie_credit_row_type", pd.Series(dtype="object")).value_counts(dropna=False).to_dict()

save_mask = goalie_df.get("goalie_credit_row_type", "").astype(str).eq("save_component")
goal_mask = goalie_df.get("goalie_credit_row_type", "").astype(str).eq("goal_penalty")

save_formula_ok = bool(np.allclose(
    goalie_df.loc[save_mask, "Net_xT"].fillna(0.0).to_numpy(),
    (goalie_df.loc[save_mask, "xT_For"].fillna(0.0) - goalie_df.loc[save_mask, "xT_Against"].fillna(0.0)).to_numpy(),
    atol=1e-10
)) if save_mask.any() else True

goal_formula_ok = bool(np.allclose(
    goalie_df.loc[goal_mask, "Net_xT"].fillna(0.0).to_numpy(),
    (goalie_df.loc[goal_mask, "xT_For"].fillna(0.0) - goalie_df.loc[goal_mask, "xT_Against"].fillna(0.0)).to_numpy(),
    atol=1e-10
)) if goal_mask.any() else True

goal_penalty_bounds_ok = bool((
    goalie_df.loc[goal_mask, "goal_penalty_xT_Against"].fillna(0.0).between(0.0, 1.0)
).all()) if goal_mask.any() else True

display(pd.DataFrame([
    {
        "save_component_rows": int(save_mask.sum()),
        "goal_penalty_rows": int(goal_mask.sum()),
        "row_type_counts": row_type_counts,
        "save_formula_ok": save_formula_ok,
        "goal_formula_ok": goal_formula_ok,
        "goal_penalty_bounds_ok": goal_penalty_bounds_ok,
    }
]))

display(goalie_df.loc[goal_mask, [
    c for c in [
        "game_id", "sl_event_id", "linked_goal_sl_event_id", "goalie_id",
        "prior_event_type_for_goal_penalty", "prior_threat_for_goal_penalty",
        "goal_penalty_xT_Against", "Net_xT", "goalie_credit_reason"
    ] if c in goalie_df.columns
]].head(15))

,save_component_rows,goal_penalty_rows,row_type_counts,save_formula_ok,goal_formula_ok,goal_penalty_bounds_ok
0,27824,2646,"{'save_component': 27824, 'goal_penalty': 2646}",True,True,True


,game_id,sl_event_id,linked_goal_sl_event_id,goalie_id,prior_event_type_for_goal_penalty,prior_threat_for_goal_penalty,goal_penalty_xT_Against,Net_xT,goalie_credit_reason
27824,00b0366a-95c6-5250-2dae-e3dd5c4198bc,293.0,293.0,24d2bb10-73c1-950e-69b2-8ed8ea9fc132,shot,0.093695,0.906305,-0.906305,goal_penalty_post_shot
27825,00b0366a-95c6-5250-2dae-e3dd5c4198bc,431.0,431.0,6eb5f6c3-b657-5df9-6027-43ce18a2cca6,shot,0.184426,0.815574,-0.815574,goal_penalty_post_shot
27826,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1384.0,1384.0,24d2bb10-73c1-950e-69b2-8ed8ea9fc132,shot,0.415000,0.585000,-0.585000,goal_penalty_post_shot
27827,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1470.0,1470.0,24d2bb10-73c1-950e-69b2-8ed8ea9fc132,shot,0.218066,0.781934,-0.781934,goal_penalty_post_shot
27828,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2091.0,2091.0,6eb5f6c3-b657-5df9-6027-43ce18a2cca6,shot,0.024442,0.975558,-0.975558,goal_penalty_post_shot
27829,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2379.0,2379.0,6eb5f6c3-b657-5df9-6027-43ce18a2cca6,shot,0.034572,0.965428,-0.965428,goal_penalty_post_shot
27830,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,569.0,569.0,9529adbf-24fa-a62b-bfdc-a3f80f06ab78,shot,0.296841,0.703159,-0.703159,goal_penalty_post_shot
27831,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1185.0,1185.0,940ea455-65b8-8157-ac39-c3623402f495,shot,0.056405,0.943595,-0.943595,goal_penalty_post_shot
27832,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,418.0,418.0,ef9fd39c-8b23-a1e2-d9a6-7170cea4e781,shot,0.028277,0.971723,-0.971723,goal_penalty_post_shot
27833,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2608.0,2608.0,ef9fd39c-8b23-a1e2-d9a6-7170cea4e781,shot,0.127473,0.872527,-0.872527,goal_penalty_post_shot


In [11]:
adj = adjusted_df.copy()
if "adjustment_source" in adj.columns:
    adj["adjustment_source"] = adj["adjustment_source"].astype(str)

for c in ["Adjusted_Net_xT", "Actor_Net_xT"]:
    if c in adj.columns:
        adj[c] = pd.to_numeric(adj[c], errors="coerce")

faceoff_base = adj.loc[adj.get("event_type", "").astype(str).str.lower().eq("faceoff"), [
    c for c in ["game_id", "sl_event_id", "Actor_Net_xT"] if c in adj.columns
]].copy()
faceoff_inv = adj.loc[adj.get("adjustment_source", "").eq("faceoff_inverse"), [
    c for c in ["game_id", "sl_event_id", "linked_kept_sl_event_id", "Adjusted_Net_xT"] if c in adj.columns
]].copy()

penalty_base = adj.loc[adj.get("event_type", "").astype(str).str.lower().eq("penalty"), [
    c for c in ["game_id", "sl_event_id", "Adjusted_Net_xT"] if c in adj.columns
]].copy()
penalty_inv = adj.loc[adj.get("adjustment_source", "").eq("penalty_drawer_inverse"), [
    c for c in ["game_id", "sl_event_id", "linked_kept_sl_event_id", "Adjusted_Net_xT"] if c in adj.columns
]].copy()

faceoff_conservation_ok = None
penalty_conservation_ok = None

if not faceoff_base.empty and not faceoff_inv.empty and "linked_kept_sl_event_id" in faceoff_inv.columns:
    fb = faceoff_base.rename(columns={"sl_event_id": "kept_sl_event_id", "Actor_Net_xT": "kept_net"})
    fi = faceoff_inv.rename(columns={"Adjusted_Net_xT": "inv_net", "linked_kept_sl_event_id": "kept_sl_event_id"})
    fi["kept_sl_event_id"] = pd.to_numeric(fi["kept_sl_event_id"], errors="coerce")
    pair = fi.merge(fb, on=["game_id", "kept_sl_event_id"], how="inner")
    if not pair.empty:
        pair["sum_net"] = pair["inv_net"].fillna(0.0) + pair["kept_net"].fillna(0.0)
        faceoff_conservation_ok = bool(np.allclose(pair["sum_net"].to_numpy(), 0.0, atol=1e-8))

if not penalty_base.empty and not penalty_inv.empty and "linked_kept_sl_event_id" in penalty_inv.columns:
    pb = penalty_base.rename(columns={"sl_event_id": "kept_sl_event_id", "Adjusted_Net_xT": "kept_net"})
    pi = penalty_inv.rename(columns={"Adjusted_Net_xT": "inv_net", "linked_kept_sl_event_id": "kept_sl_event_id"})
    pi["kept_sl_event_id"] = pd.to_numeric(pi["kept_sl_event_id"], errors="coerce")
    pair = pi.merge(pb, on=["game_id", "kept_sl_event_id"], how="inner")
    if not pair.empty:
        pair["sum_net"] = pair["inv_net"].fillna(0.0) + pair["kept_net"].fillna(0.0)
        penalty_conservation_ok = bool(np.allclose(pair["sum_net"].to_numpy(), 0.0, atol=1e-8))

display(pd.DataFrame([
    {
        "faceoff_inverse_rows": int((adj.get("adjustment_source", "").eq("faceoff_inverse")).sum()) if "adjustment_source" in adj.columns else 0,
        "penalty_drawer_inverse_rows": int((adj.get("adjustment_source", "").eq("penalty_drawer_inverse")).sum()) if "adjustment_source" in adj.columns else 0,
        "faceoff_conservation_ok": faceoff_conservation_ok,
        "penalty_conservation_ok": penalty_conservation_ok,
    }
]))

,faceoff_inverse_rows,penalty_drawer_inverse_rows,faceoff_conservation_ok,penalty_conservation_ok
0,28091,3408,None,None


In [12]:
summary_rows = summary.get("rows", {}) if isinstance(summary.get("rows"), dict) else {}
goalie_counts = summary_rows.get("goalie_row_type_counts", {}) if isinstance(summary_rows.get("goalie_row_type_counts"), dict) else {}
goal_pred_warn = summary.get("goal_predecessor_warning_audit", {}) if isinstance(summary.get("goal_predecessor_warning_audit"), dict) else {}
goalie_goal_audit = summary.get("goalie_goal_penalty_audit", {}) if isinstance(summary.get("goalie_goal_penalty_audit"), dict) else {}
tensor_linkage_audit = summary.get("tensor_linkage_audit", {}) if isinstance(summary.get("tensor_linkage_audit"), dict) else {}

tensor_goal_rows = int(tensor_linkage_audit.get("tensor_goal_rows", 0) or 0)
resolved_goal_penalty_rows = int(goalie_counts.get("goal_penalty", 0) or 0)

checks = {
    "summary_matches_adjusted_rows": int(summary_rows.get("adjusted", -1)) == int(len(adjusted_df)),
    "summary_matches_goalie_rows": int(summary_rows.get("goalie_ledger", -1)) == int(len(goalie_df)),
    "save_component_rows_present": int(goalie_counts.get("save_component", 0)) > 0,
    "goal_predecessor_audit_present": bool(goal_pred_warn),
    "tensor_goal_population_present": tensor_goal_rows > 0,
    "goal_penalty_rows_present": resolved_goal_penalty_rows > 0,
}

final_status = "PASS" if all(checks.values()) else "WARN"
display(pd.DataFrame([
    {
        "run_label": RUN_LABEL,
        "final_status": final_status,
        "tensor_goal_rows": tensor_goal_rows,
        "goal_penalty_rows": resolved_goal_penalty_rows,
        **checks,
    }
]))

if goal_pred_warn:
    display(pd.DataFrame([
        {
            "population_source": goal_pred_warn.get("population_source"),
            "goal_rows_total": goal_pred_warn.get("goal_rows_total"),
            "canonical_goal_rows_total": goal_pred_warn.get("canonical_goal_rows_total"),
            "unexpected_predecessor_rows": goal_pred_warn.get("unexpected_predecessor_rows"),
            "unexpected_predecessor_rate": goal_pred_warn.get("unexpected_predecessor_rate"),
            "policy": goal_pred_warn.get("policy"),
        }
    ]))

if goalie_goal_audit:
    display(pd.DataFrame([
        {
            "goal_penalty_population_source": goalie_goal_audit.get("goal_penalty_population_source"),
            "tensor_goal_rows_total": goalie_goal_audit.get("tensor_goal_rows_total"),
            "tensor_goal_rows_with_linked_source": goalie_goal_audit.get("tensor_goal_rows_with_linked_source"),
            "tensor_goal_rows_with_resolved_goalie": goalie_goal_audit.get("tensor_goal_rows_with_resolved_goalie"),
            "goal_penalty_rows_emitted": goalie_goal_audit.get("goal_penalty_rows_emitted"),
            "goal_penalty_unresolved_linkage_rows": goalie_goal_audit.get("goal_penalty_unresolved_linkage_rows"),
            "goal_penalty_unresolved_goalie_rows": goalie_goal_audit.get("goal_penalty_unresolved_goalie_rows"),
        }
    ]))

,run_label,final_status,tensor_goal_rows,goal_penalty_rows,summary_matches_adjusted_rows,summary_matches_goalie_rows,save_component_rows_present,goal_predecessor_audit_present,tensor_goal_population_present,goal_penalty_rows_present
0,Events_With_Saves,PASS,2816,2646,True,True,True,True,True,True


,population_source,goal_rows_total,canonical_goal_rows_total,unexpected_predecessor_rows,unexpected_predecessor_rate,policy
0,tensor_goal_rows,0,3079,0,0.0,warn_only


,goal_penalty_population_source,tensor_goal_rows_total,tensor_goal_rows_with_linked_source,tensor_goal_rows_with_resolved_goalie,goal_penalty_rows_emitted,goal_penalty_unresolved_linkage_rows,goal_penalty_unresolved_goalie_rows
0,tensor_goal_rows_only,2815,2815,2646,2646,0,169
